<a href="https://colab.research.google.com/github/TanyaSrivastava444/marketing-mix-model-Staffing/blob/main/Notebooks/Data_generator_MMM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:

import numpy as np
import pandas as pd
from fredapi import Fred

np.random.seed(42)

In [35]:
pip install fredapi

In [36]:
# 1 DEFINE DATE RANGE


weeks = pd.date_range(
    start="2021-01-03",
    end="2023-06-25",
    freq="W"
)

n = len(weeks)


In [37]:
# 2 LOAD BLS MACRO DATA


fred = Fred(api_key="7d8b408743556636f900ec85885bd8d0")

unemployment = fred.get_series("UNRATE")
job_openings = fred.get_series("JTSJOL")
payrolls = fred.get_series("PAYEMS")

macro = pd.concat({
    "unemployment_rate": unemployment,
    "job_openings": job_openings,
    "payrolls": payrolls,
}, axis=1, join="inner")

macro.index.name = "date"
macro = macro.reset_index()
macro = macro[(macro["date"] >= "2021-01-01") & (macro["date"] <= "2023-06-30")]

In [39]:
# 3 CREATE BASE DATAFRAME
###############################################

data = pd.DataFrame({
    "week": weeks
})


In [40]:
# Convert monthly macro data to weekly
macro_weekly = (
    macro
    .set_index("date")
    .resample("W")
    .ffill()
    .reset_index()
)

# Merge with synthetic dataset
data = data.merge(
    macro_weekly,
    left_on="week",
    right_on="date",
    how="left"
)

# Remove redundant column
data = data.drop(columns=["date"])

In [42]:
data.tail()

,week,unemployment_rate,job_openings,payrolls
125,2023-05-28,3.6,9310.0,155655.0
126,2023-06-04,3.6,9191.0,155880.0
127,2023-06-11,NaN,NaN,NaN
128,2023-06-18,NaN,NaN,NaN
129,2023-06-25,NaN,NaN,NaN


In [43]:
# 4 CREATE SEASONALITY
###############################################

t = np.arange(n)

data["sin_season"] = np.sin(2*np.pi*t/52)
data["cos_season"] = np.cos(2*np.pi*t/52)

In [44]:
# 5 GENERATE MARKETING SPEND
###############################################

def generate_spend(base, volatility):

    trend = np.linspace(0, base*0.3, n)

    seasonal = base * (
        1 + 0.2*np.sin(2*np.pi*t/52)
    )

    noise = np.random.normal(0, volatility, n)

    return seasonal + trend + noise


data["job_board_spend"] = generate_spend(60000,7000)
data["linkedin_spend"] = generate_spend(45000,6000)
data["events_spend"] = generate_spend(20000,5000)
data["youtube_ads_spend"] = generate_spend(15000,3000)
data["google_ads_spend"] = generate_spend(30000,4000)
data["client_outreach_spend"] = generate_spend(10000,2000)


In [46]:
data.tail(5)

,week,unemployment_rate,job_openings,payrolls,sin_season,cos_season,job_board_spend,linkedin_spend,events_spend,youtube_ads_spend,google_ads_spend,client_outreach_spend
125,2023-05-28,3.6,9310.0,155655.0,0.568065,-0.822984,99591.826807,60288.573632,28838.181408,23330.833034,41038.424434,10902.656798
126,2023-06-04,3.6,9191.0,155880.0,0.464723,-0.885456,76224.319137,69970.021955,29428.137683,22292.269916,30791.490135,12405.404551
127,2023-06-11,NaN,NaN,NaN,0.354605,-0.935016,78012.104770,57236.124864,36706.250488,17561.381485,40770.914972,13167.660875
128,2023-06-18,NaN,NaN,NaN,0.239316,-0.970942,81429.812643,63212.106385,31662.870220,20481.060188,39442.388423,13306.508656
129,2023-06-25,NaN,NaN,NaN,0.120537,-0.992709,75922.110584,64232.634443,23597.628443,22115.771411,42508.045541,14482.417556


In [47]:
print(data.columns.tolist())

['week', 'unemployment_rate', 'job_openings', 'payrolls', 'sin_season', 'cos_season', 'job_board_spend', 'linkedin_spend', 'events_spend', 'youtube_ads_spend', 'google_ads_spend', 'client_outreach_spend']


In [48]:
# 6 ADSTOCK FUNCTION
###############################################

def adstock(x, decay):

    result = np.zeros(len(x))

    for i in range(len(x)):
        if i == 0:
            result[i] = x[i]
        else:
            result[i] = x[i] + decay * result[i-1]

    return result

In [49]:

data["linkedin_effect"] = adstock(data["linkedin_spend"],0.5)
data["jobboard_effect"] = adstock(data["job_board_spend"],0.4)
data["events_effect"] = adstock(data["events_spend"],0.6)
data["youtube_effect"] = adstock(data["youtube_ads_spend"],0.5)
data["google_effect"] = adstock(data["google_ads_spend"],0.3)
data["outreach_effect"] = adstock(data["client_outreach_spend"],0.4)


In [50]:
# 7 SATURATION (DIMINISHING RETURNS)
###############################################

def hill(x, alpha, gamma):

    return (x**alpha) / (x**alpha + gamma**alpha)


data["linkedin_trans"] = hill(data["linkedin_effect"],1.5,50000)
data["jobboard_trans"] = hill(data["jobboard_effect"],1.3,70000)
data["events_trans"] = hill(data["events_effect"],1.7,20000)
data["youtube_trans"] = hill(data["youtube_effect"],1.4,15000)
data["google_trans"] = hill(data["google_effect"],1.3,35000)
data["outreach_trans"] = hill(data["outreach_effect"],1.2,12000)

In [52]:
###############################################
# 8 CREATE PLACEMENTS
###############################################

macro_factor = (
    0.8*(data["job_openings"]/10000)
    - 1.2*data["unemployment_rate"]
    + 0.00001*data["payrolls"]
)

marketing_effect = (
    40*data["linkedin_trans"]
    + 35*data["jobboard_trans"]
    + 20*data["events_trans"]
    + 18*data["youtube_trans"]
    + 25*data["google_trans"]
    + 12*data["outreach_trans"]
)

seasonal_effect = 15*data["sin_season"]

noise = np.random.normal(0,3,n)

data["placements"] = (
    marketing_effect
    + macro_factor
    + seasonal_effect
    + noise
)

In [54]:
# FINAL DATASET
###############################################

dataset = data[[
"week",
"job_board_spend",
"linkedin_spend",
"events_spend",
"youtube_ads_spend",
"google_ads_spend",
"client_outreach_spend",
"unemployment_rate",
"job_openings",
"payrolls",
"placements"
]]

dataset.to_csv("synthetic_staffing_mmm_data.csv",index=False)

print(dataset.head())

        week  job_board_spend  linkedin_spend  events_spend  \
0 2021-01-03     63476.999071    35696.019414  15365.347642   
1 2021-01-10     60618.124939    46600.859134  20231.031569   
2 2021-01-17     67684.677506    40989.321022   4843.949213   
3 2021-01-24     75335.072291    51346.952056  16436.016225   
4 2021-01-31     64495.743976    44084.567794  20782.098443   

   youtube_ads_spend  google_ads_spend  client_outreach_spend  \
0       12304.755986      23322.378876           13697.912190   
1       16872.251276      32966.428293           12517.459234   
2       11827.014814      28924.933834            9987.365575   
3       21662.842121      34619.326322            8565.925398   
4       20072.024762      30014.372174           16169.189206   

   unemployment_rate  job_openings  payrolls  placements  
0                6.4        7204.0  142863.0   57.604871  
1                6.4        7204.0  142863.0   81.520929  
2                6.4        7204.0  142863.0   90.073